In [ ]:
!pip install -U bitsandbytes>=0.46.1
!pip install -U accelerate
!pip install trl
!pip install peft

In [1]:
import os
import warnings
from dotenv import load_dotenv
load_dotenv(override=True)

from google.colab import drive
drive.mount('/content/drive')
base_path = "/content/drive/MyDrive/Colab Notebooks/Quant"
os.chdir(base_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from huggingface_hub import login
hf_token = os.environ.get("HUGGINGFACE_KEY")
login(token=hf_token)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [3]:
import torch 
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, GenerationConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16 
)

tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b")
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2b",
    quantization_config=quantization_config,
    device_map="auto"
)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [4]:
template = "{instruction}\n\nDecisions:"
instruction = """
You are a financial decision assistant.

Your task is to decide whether to BUY, HOLD, or SELL a stock based on the following information:
1. LSTM model output (Estimated return for today based on stock price history of the past 60 days)
2. BERT model output (Analyzes recent 100 news article)
3. Current macroeconomic conditions
4. Current portfolio status

This is the INPUT you should refer to:
{input_str}

You must provide a concise decision.

Rules (You MUST MUST MUST Follow these rules. Otherwise, I might die.):
- Your answer must contain ONLY:
  1) Decision: BUY / HOLD / SELL (Choose only one)
  2) Amount of Sharese to Trade
  3) Reason: one or two short sentences explaining the main factors.
- Do NOT provide extra commentary.
- Do NOT explain step-by-step reasoning.
- Stop immediately after the reason.

Output format:

Decision: <BUY | HOLD | SELL> (Choose one based on your reasoning)
Shares: <integer> (How much share you want me to trade)
Reason: <brief explanation>

END RESPONDING

Output Example:

Decision: BUY
Shares: 10 Shares
Reason: Since both LSTM result and the Sentiment Analysis results are good, the stock is expected to increase
"""

lstm_output = "LSTM model output : -0.25%"
bert_output = "BERT model output : 78% Positive, 12% Neutral, 10% Negative"
macro_data = "Inflation Rate Higher than normal"
open_price = 312
cash = 30043
shares_owned = 30


input_str = f"""
LSTM Prediction: {lstm_output}
BERT Sentiment: {bert_output}
Inflation / Macro Conditions: {macro_data}
Opening Price: ${open_price}
Cash: ${cash}
Current Holdings: {shares_owned} shares
"""

instruction = instruction.format(input_str=input_str)

prompt = template.format(
    instruction=instruction,
)

input_ids = tokenizer(prompt, return_tensors="pt").to(model.device)
input_len = input_ids.input_ids.shape[1]

config = GenerationConfig(
    do_sample=False,
    max_new_tokens=50, 
    repetition_penalty=1.5,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id
)

outputs = model.generate(**input_ids, generation_config=config)
new_response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

print(new_response.strip())

[HOLD]
[SHARE]: 100000000000000000000000000000000000000000


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
import os, json
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=16,              
    lora_alpha=32,    
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"], 
    lora_dropout=0.05,   
    bias="none",
    task_type="CAUSAL_LM",
)

with open(os.path.join("data", "gemma_finetune.json"), 'r', encoding='utf-8') as file:
    data_list = json.load(file)

texts = [
    f"<start_of_turn>user\n{item['instruction']}\n\nInput: {item['input']}<end_of_turn>\n<start_of_turn>model\n{item['output']}<end_of_turn>"
    for item in data_list
]

from datasets import Dataset
dataset = Dataset.from_dict({"text": texts})

training_args = TrainingArguments(
    output_dir="./gemma-finetuned",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_steps=500,
    logging_steps=10,
    fp16=False,
    bf16=True,
    remove_unused_columns=False 
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    peft_config=peft_config,
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Step,Training Loss
10,1.979621


In [ ]:
new_model_path = "gemma-finetuned-final"

trainer.model.save_pretrained(new_model_path)
tokenizer.save_pretrained(new_model_path)

('gemma-finetuned-final/tokenizer_config.json',
 'gemma-finetuned-final/tokenizer.json')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

base_model_id = "google/gemma-2b"

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, new_model_path)

model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GemmaForCausalLM(
      (model): GemmaModel(
        (embed_tokens): Embedding(256000, 2048, padding_idx=0)
        (layers): ModuleList(
          (0-17): 18 x GemmaDecoderLayer(
            (self_attn): GemmaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj)

In [ ]:
template = "{instruction}\n\nDecisions:"
instruction = """
You are a financial decision assistant.

Your task is to decide whether to BUY, HOLD, or SELL a stock based on the following information:
1. LSTM model output (Estimated return for today based on stock price history of the past 60 days)
2. BERT model output (Analyzes recent 100 news article)
3. Current macroeconomic conditions
4. Current portfolio status

This is the INPUT you should refer to:
{input_str}

You must provide a concise decision.

Rules (You MUST MUST MUST Follow these rules. Otherwise, I might die.):
- Your answer must contain ONLY:
  1) Decision: BUY / HOLD / SELL (Choose only one)
  2) Amount of Sharese to Trade
  3) Reason: one or two short sentences explaining the main factors.
- Do NOT provide extra commentary.
- Do NOT explain step-by-step reasoning.
- Stop immediately after the reason.

Output format:

Decision: <BUY | HOLD | SELL> (Choose one based on your reasoning)
Shares: <integer> (How much share you want me to trade)
Reason: <brief explanation>

END RESPONDING

Output Example:

Decision: BUY
Shares: 10 Shares
Reason: Since both LSTM result and the Sentiment Analysis results are good, the stock is expected to increase
"""

lstm_output = "LSTM model output : -6.25%"
bert_output = "BERT model output : 78% Negative, 12% Neutral, 10% Negative"
macro_data = "Inflation Rate Higher than normal"
open_price = 312
cash = 30043
shares_owned = 30


input_str = f"""
LSTM Prediction: {lstm_output}
BERT Sentiment: {bert_output}
Inflation / Macro Conditions: {macro_data}
Opening Price: ${open_price}
Cash: ${cash}
Current Holdings: {shares_owned} shares
"""

instruction = instruction.format(input_str=input_str)

prompt = template.format(
    instruction=instruction,
)

input_ids = tokenizer(prompt, return_tensors="pt").to(model.device)
input_len = input_ids.input_ids.shape[1]

config = GenerationConfig(
    do_sample=False,
    max_new_tokens=50, 
    repetition_penalty=1.5,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id
)

outputs = model.generate(**input_ids, generation_config=config)
new_response = tokenizer.decode(outputs[0][:], skip_special_tokens=True)

print(new_response.strip())

You are a financial decision assistant.

Your task is to decide whether to BUY, HOLD, or SELL a stock based on the following information:
1. LSTM model output (Estimated return for today based on stock price history of the past 60 days)
2. BERT model output (Analyzes recent 100 news article)
3. Current macroeconomic conditions
4. Current portfolio status

This is the INPUT you should refer to:

LSTM Prediction: LSTM model output : -6.25%
BERT Sentiment: BERT model output : 78% Negative, 12% Neutral, 10% Negative
Inflation / Macro Conditions: Inflation Rate Higher than normal
Opening Price: $312
Cash: $30043
Current Holdings: 30 shares


You must provide a concise decision.

Rules (You MUST MUST MUST Follow these rules. Otherwise, I might die.):
- Your answer must contain ONLY:
  1) Decision: BUY / HOLD / SELL (Choose only one)
  2) Amount of Sharese to Trade
  3) Reason: one or two short sentences explaining the main factors.
- Do NOT provide extra commentary.
- Do NOT explain step-by-